# ChurnIQ | Exploratory data analysis
This notebook describes the IBM Telco sample. Its figures summarize observed associations. The training command creates its own stratified split and estimates preprocessing within each training fold. For a new modeling iteration driven by EDA, reserve a fresh evaluation set or limit exploratory work to the training partition.

Run `python -m scripts.download_data` from the project root before opening the notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")
root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent
raw = pd.read_csv(root / "data/raw/Telco-Customer-Churn.csv", keep_default_na=False)
print(f"{len(raw):,} customer rows and {len(raw.columns)} columns")
display(raw.drop(columns="customerID").head())

## Data dictionary

| Columns | Meaning |
| --- | --- |
| customerID | Unique customer key excluded from model features |
| gender, SeniorCitizen, Partner, Dependents | Customer demographics and household indicators |
| tenure | Months with the provider |
| PhoneService, MultipleLines | Telephone service and line count category |
| InternetService | DSL, fiber optic, or no internet |
| OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport | Internet add-on subscriptions and eligibility |
| StreamingTV, StreamingMovies | Streaming service enrollment and eligibility |
| Contract, PaperlessBilling, PaymentMethod | Account term and billing preferences |
| MonthlyCharges, TotalCharges | Current monthly and accumulated charges |
| Churn | Whether the customer left, recorded as Yes or No |

The sample uses explicit no-service categories. A blank field represents missing information.

In [ ]:
data = raw.replace(r"^\s*$", pd.NA, regex=True)
print("Duplicate full rows:", raw.duplicated().sum())
print("Duplicate customer IDs:", raw.customerID.duplicated().sum())
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
display(pd.DataFrame({"raw dtype": raw.dtypes, "analysis dtype": data.dtypes,
                      "missing": data.isna().sum()}))
display(data.loc[data.TotalCharges.isna(), "tenure"].value_counts().rename("Missing total charges"))
data["churn_flag"] = data.Churn.eq("Yes").astype(int)

The blank total charges occur at zero tenure. A prospective model can use a training median for missing amounts while retaining tenure as a separate input. Deciding that the true charge was zero would require a source definition.

In [ ]:
distribution = data.Churn.value_counts().to_frame("customers")
distribution["share"] = distribution.customers / len(data)
display(distribution)
print(f"Observed churn rate: {data.churn_flag.mean():.2%}")
sns.countplot(data=data, x="Churn", order=["No", "Yes"])
plt.title("Observed churn classes")
plt.show()

The positive class is a minority. An always-stay classifier would appear accurate in many cases while missing every churner. This motivates metrics such as recall, ROC-AUC and average precision.

In [ ]:
contract = (data.groupby("Contract").churn_flag.agg(["size", "mean"])
            .rename(columns={"size": "customers", "mean": "churn_rate"})
            .sort_values("churn_rate", ascending=False))
display(contract.style.format({"churn_rate": "{:.1%}"}))
contract.churn_rate.plot.bar(rot=15, ylabel="Fraction who churned", ylim=(0, 1))
plt.show()

Month-to-month customers have the highest observed churn share. Contract term can overlap with tenure, customer self-selection and other factors, so the difference cannot identify the effect of changing contracts.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=data, x="Churn", y="MonthlyCharges", ax=axes[0])
sns.histplot(data=data, x="tenure", hue="Churn", stat="percent", common_norm=False,
             element="step", bins=18, ax=axes[1])
plt.tight_layout()
plt.show()
display(data.groupby("Churn")[["MonthlyCharges", "tenure"]].median())

Customers who left show shorter typical tenure and higher median monthly charges in this sample. The distributions overlap, so either field alone leaves substantial uncertainty.

In [ ]:
for field in ["InternetService", "OnlineSecurity", "TechSupport"]:
    print(field)
    table = data.groupby(field).churn_flag.agg(customers="size", churn_rate="mean")
    display(table.style.format({"churn_rate": "{:.1%}"}))
eligible = data[data.InternetService.ne("No")].copy()
sns.barplot(data=eligible, x="TechSupport", y="churn_flag", hue="InternetService",
            errorbar=None)
plt.ylabel("Observed churn rate")
plt.show()

Service eligibility matters when comparing add-ons. The plot compares support enrollment within each internet technology among internet subscribers. Group rates can still vary with account and customer characteristics.

In [ ]:
features = ["tenure", "MonthlyCharges", "TotalCharges", "churn_flag"]
correlations = data[features].corr()
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1)
plt.title("Pairwise Pearson correlations")
plt.tight_layout()
plt.show()

Tenure and total charges are related because total charges accumulate over time. Pearson correlation summarizes linear association. The correlation with binary churn reflects differences between the two groups and gives a limited view of nonlinear effects or interactions. Check correlated predictors when discussing feature importance.